# 02 · Rolling-origin backtest of day-ahead forecasts

**Task:** at midnight, forecast the average speed of every modelled cell for each hour of the next day.

**Protocol:** for every test week, retrain on all data strictly *before* it, then forecast day by day. All target-derived features are lagged by at least 24 h, and a unit test perturbs the last 24 h and checks that the features don't change. There are 10 test weeks: 8 consecutive weeks in autumn 2024 (including Republic Day and the November school break) and the two Bayram weeks.

| Model | Description |
|---|---|
| Seasonal naive (day) | same hour yesterday |
| Seasonal naive (week) | same hour last week, a strong baseline for traffic |
| LightGBM without calendar | lags, rolling stats, hour/weekday, cell |
| **LightGBM (full)** | + public/religious holidays, eves, school calendar, distance to holidays |

In [1]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from istanbul_traffic import plots
from istanbul_traffic.calendar import calendar_features

plots.apply_style()
pd.set_option("display.float_format", "{:,.3f}".format)
metrics = json.loads(Path("reports/metrics.json").read_text())
panel = pd.read_parquet("data/processed/panel.parquet")
cal = calendar_features(pd.DatetimeIndex(panel["date_time"].unique()))
panel = panel.join(cal, on="date_time")
cat = pd.read_csv("reports/cell_catalogue.csv", index_col=0)

In [2]:
pd.read_csv("reports/backtest_overall.csv", index_col=0)

,mae,rmse,wape,skill_vs_naive_week,n
Seasonal naive (same hour yesterday),6.668,10.400,0.119,-0.008,"239,850.000"
Seasonal naive (same hour last week),6.615,10.569,0.118,0.000,"239,850.000"
LightGBM without holiday/school features,4.689,7.144,0.084,0.291,"239,850.000"
LightGBM (full),4.312,6.483,0.077,0.348,"239,850.000"


In [3]:
s = metrics["scores"]
rows = {
    k: {m: v["mae"] for m, v in s[k].items()}
    for k in ("regular_weeks", "holiday_weeks", "holiday_days_only")
}
pd.DataFrame(rows).rename(
    columns={
        "regular_weeks": "MAE regular weeks",
        "holiday_weeks": "MAE Bayram weeks",
        "holiday_days_only": "MAE holiday days only",
    }
)

,MAE regular weeks,MAE Bayram weeks,MAE holiday days only
Seasonal naive (same hour yesterday),6.740,6.325,7.211
Seasonal naive (same hour last week),5.282,12.925,14.134
LightGBM without holiday/school features,4.095,7.500,8.436
LightGBM (full),3.886,6.332,6.734


**Reading the results**

* **Ordinary weeks:** LightGBM cuts the error of the strongest baseline (same hour last week) by **26 %** (MAE 3.9 vs 5.3 km/h). "Same hour yesterday" is much worse because Monday is not like Sunday.
* **Bayram weeks:** the weekly naive forecast collapses (MAE 12.9). During a multi-day holiday, *yesterday* is a good guide, and the full LightGBM ties with it on the whole week (6.33 vs 6.33). On the **holiday days themselves** it is the best model (6.7 vs 7.2 for yesterday and 14.1 for last week).
* **Calendar features matter exactly where expected:** removing them raises the error on holiday days by 25 % (6.7 → 8.4) and on school-break weekdays by 18 % (4.5 → 5.3), while ordinary days barely change.

![folds](../reports/figures/backtest_by_fold.png)

In [4]:
bt = pd.DataFrame(
    {t: {m: v["mae"] for m, v in d.items()} for t, d in metrics["scores"]["by_day_type"].items()}
)
bt

,holiday,weekday (school break),weekday (school day),weekend
Seasonal naive (same hour yesterday),7.211,5.670,6.146,7.860
Seasonal naive (same hour last week),14.134,8.129,5.109,5.526
LightGBM without holiday/school features,8.436,5.277,3.914,4.268
LightGBM (full),6.734,4.490,3.708,4.325


## Experiment: does a second year of history help?

Bayrams move about 11 days earlier every year, so a model trained only on 2024 has **never seen a Bayram** before the Ramazan test week. Re-running the same backtest with 2024 data only (`uv run traffic-backtest --months 202401 ... 202412`) gave:

| LightGBM (full), MAE km/h | 2024 only | 2023 + 2024 |
|---|---|---|
| Ramazan Bayramı week | 8.66 | **5.95** |
| Kurban Bayramı week | 8.76 | **7.32** |
| Ordinary weeks (avg.) | 3.95 | **3.89** |

One extra year of history, which contains the previous year's Bayrams, removes most of the holiday penalty. A production system should therefore keep **several years** of training data, even though ordinary-week accuracy hardly changes.

## Where does the error come from?

![hour](../reports/figures/error_by_hour.png)

Errors are highest in the peaks, when small changes in demand cause large changes in speed. The calendar features help most in the **morning peak (06–10 h)**, which is driven by school traffic.

## A Bayram week, hour by hour

![kurban](../reports/figures/forecast_kurban_bayrami.png)

"Same hour last week" predicts a normal working-day rush hour on Bayram days, which never comes. LightGBM knows that these days are holidays and follows the much lighter traffic.

## Congestion alerts

A cell-hour counts as *congested* when its speed is below 60 % of its free-flow (night) speed (≈ 30 % of cell-hours).

In [5]:
pd.DataFrame(metrics["congestion_alerts_regular_weeks"]).T

,precision,recall,f1,congested_share
Seasonal naive (same hour last week),0.802,0.801,0.802,0.303
LightGBM without holiday/school features,0.879,0.815,0.846,0.303
LightGBM (full),0.884,0.833,0.858,0.303


Used as a day-ahead **congestion alert**, the model is right 88 % of the time it raises an alert (precision) and catches 83 % of congested hours (recall), for an F1 of 0.86 vs 0.80 for the weekly naive forecast.

## What drives the model?

![importance](../reports/figures/feature_importance.png)

In [6]:
pd.Series(metrics["top_features"], name="mean |SHAP| (km/h)").to_frame()

,mean |SHAP| (km/h)
same_hour_4w_mean,9.465
lag_24,1.833
hour,1.684
lag_168,1.349
cell,1.161
dow,0.805
lag_48,0.556
lag_72,0.553
roll24_mean,0.514
roll168_mean,0.316


The 4-week same-hour average dominates: traffic is above all a strong weekly rhythm. Yesterday's value, the hour and the cell's identity then adjust it. Holiday features rank low *on average* because holidays are rare. Where they apply, they are decisive (see the ablation above).

## Limitations & next steps

* Only the 150 busiest well-covered cells are modelled. Quieter cells are noisier and would need hierarchical pooling.
* No weather, events, accidents or road-works data; these drive most of the remaining peak-hour error.
* Two years of history, so each Bayram type has been seen only once before the test. More years would help further.
* Next: probabilistic forecasts (quantile LightGBM) for alert thresholds, and a spatial model that uses neighbouring cells.